In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy
import sys
import os
import pickle
import librosa
import librosa.display
from IPython.display import Audio
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import tensorflow as tf
from tensorflow import keras

In [5]:
df = pd.read_csv('Data/features_3_sec.csv')

In [6]:
df.head()

,filename,length,chroma_stft_mean,chroma_stft_var,rms_mean,rms_var,spectral_centroid_mean,spectral_centroid_var,spectral_bandwidth_mean,spectral_bandwidth_var,...,mfcc16_var,mfcc17_mean,mfcc17_var,mfcc18_mean,mfcc18_var,mfcc19_mean,mfcc19_var,mfcc20_mean,mfcc20_var,label
0,blues.00000.0.wav,66149,0.335406,0.091048,0.130405,0.003521,1773.065032,167541.630869,1972.744388,117335.771563,...,39.687145,-3.241280,36.488243,0.722209,38.099152,-5.050335,33.618073,-0.243027,43.771767,blues
1,blues.00000.1.wav,66149,0.343065,0.086147,0.112699,0.001450,1816.693777,90525.690866,2010.051501,65671.875673,...,64.748276,-6.055294,40.677654,0.159015,51.264091,-2.837699,97.030830,5.784063,59.943081,blues
2,blues.00000.2.wav,66149,0.346815,0.092243,0.132003,0.004620,1788.539719,111407.437613,2084.565132,75124.921716,...,67.336563,-1.768610,28.348579,2.378768,45.717648,-1.938424,53.050835,2.517375,33.105122,blues
3,blues.00000.3.wav,66149,0.363639,0.086856,0.132565,0.002448,1655.289045,111952.284517,1960.039988,82913.639269,...,47.739452,-3.841155,28.337118,1.218588,34.770935,-3.580352,50.836224,3.630866,32.023678,blues
4,blues.00000.4.wav,66149,0.335579,0.088129,0.143289,0.001701,1630.656199,79667.267654,1948.503884,60204.020268,...,30.336359,0.664582,45.880913,1.689446,51.363583,-3.392489,26.738789,0.536961,29.146694,blues


In [7]:
df.shape

(9990, 60)

In [8]:
df.dtypes

filename                    object
length                       int64
chroma_stft_mean           float64
chroma_stft_var            float64
rms_mean                   float64
rms_var                    float64
spectral_centroid_mean     float64
spectral_centroid_var      float64
spectral_bandwidth_mean    float64
spectral_bandwidth_var     float64
rolloff_mean               float64
rolloff_var                float64
zero_crossing_rate_mean    float64
zero_crossing_rate_var     float64
harmony_mean               float64
harmony_var                float64
perceptr_mean              float64
perceptr_var               float64
tempo                      float64
mfcc1_mean                 float64
mfcc1_var                  float64
mfcc2_mean                 float64
mfcc2_var                  float64
mfcc3_mean                 float64
mfcc3_var                  float64
mfcc4_mean                 float64
mfcc4_var                  float64
mfcc5_mean                 float64
mfcc5_var           

In [9]:
df.drop(labels='filename', axis=1)

,length,chroma_stft_mean,chroma_stft_var,rms_mean,rms_var,spectral_centroid_mean,spectral_centroid_var,spectral_bandwidth_mean,spectral_bandwidth_var,rolloff_mean,...,mfcc16_var,mfcc17_mean,mfcc17_var,mfcc18_mean,mfcc18_var,mfcc19_mean,mfcc19_var,mfcc20_mean,mfcc20_var,label
0,66149,0.335406,0.091048,0.130405,0.003521,1773.065032,167541.630869,1972.744388,117335.771563,3714.560359,...,39.687145,-3.241280,36.488243,0.722209,38.099152,-5.050335,33.618073,-0.243027,43.771767,blues
1,66149,0.343065,0.086147,0.112699,0.001450,1816.693777,90525.690866,2010.051501,65671.875673,3869.682242,...,64.748276,-6.055294,40.677654,0.159015,51.264091,-2.837699,97.030830,5.784063,59.943081,blues
2,66149,0.346815,0.092243,0.132003,0.004620,1788.539719,111407.437613,2084.565132,75124.921716,3997.639160,...,67.336563,-1.768610,28.348579,2.378768,45.717648,-1.938424,53.050835,2.517375,33.105122,blues
3,66149,0.363639,0.086856,0.132565,0.002448,1655.289045,111952.284517,1960.039988,82913.639269,3568.300218,...,47.739452,-3.841155,28.337118,1.218588,34.770935,-3.580352,50.836224,3.630866,32.023678,blues
4,66149,0.335579,0.088129,0.143289,0.001701,1630.656199,79667.267654,1948.503884,60204.020268,3469.992864,...,30.336359,0.664582,45.880913,1.689446,51.363583,-3.392489,26.738789,0.536961,29.146694,blues
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9985,66149,0.349126,0.080515,0.050019,0.000097,1499.083005,164266.886443,1718.707215,85931.574523,3015.559458,...,42.485981,-9.094270,38.326839,-4.246976,31.049839,-5.625813,48.804092,1.818823,38.966969,rock
9986,66149,0.372564,0.082626,0.057897,0.000088,1847.965128,281054.935973,1906.468492,99727.037054,3746.694524,...,32.415203,-12.375726,66.418587,-3.081278,54.414265,-11.960546,63.452255,0.428857,18.697033,rock
9987,66149,0.347481,0.089019,0.052403,0.000701,1346.157659,662956.246325,1561.859087,138762.841945,2442.362154,...,78.228149,-2.524483,21.778994,4.809936,25.980829,1.775686,48.582378,-0.299545,41.586990,rock
9988,66149,0.387527,0.084815,0.066430,0.000320,2084.515327,203891.039161,2018.366254,22860.992562,4313.266226,...,28.323744,-5.363541,17.209942,6.462601,21.442928,2.354765,24.843613,0.675824,12.787750,rock


In [10]:
audio_recording = 'Data/genres_original/hiphop/hiphop.00004.wav'
data, sr = librosa.load(audio_recording)
librosa.load(audio_recording, sr = 45600)


(array([ 0.27448815,  0.3246269 ,  0.30336043, ..., -0.30028528,
        -0.18890929,  0.        ], dtype=float32),
 45600)

In [14]:
Audio(data, rate=sr)

In [23]:
class_list = df.iloc[:,-1]
convertor = LabelEncoder()

In [24]:
y = convertor.fit_transform(class_list)
y

array([0, 0, 0, ..., 9, 9, 9])

In [27]:
df.iloc[:,:-1]

,filename,length,chroma_stft_mean,chroma_stft_var,rms_mean,rms_var,spectral_centroid_mean,spectral_centroid_var,spectral_bandwidth_mean,spectral_bandwidth_var,...,mfcc16_mean,mfcc16_var,mfcc17_mean,mfcc17_var,mfcc18_mean,mfcc18_var,mfcc19_mean,mfcc19_var,mfcc20_mean,mfcc20_var
0,blues.00000.0.wav,66149,0.335406,0.091048,0.130405,0.003521,1773.065032,167541.630869,1972.744388,117335.771563,...,-2.853603,39.687145,-3.241280,36.488243,0.722209,38.099152,-5.050335,33.618073,-0.243027,43.771767
1,blues.00000.1.wav,66149,0.343065,0.086147,0.112699,0.001450,1816.693777,90525.690866,2010.051501,65671.875673,...,4.074709,64.748276,-6.055294,40.677654,0.159015,51.264091,-2.837699,97.030830,5.784063,59.943081
2,blues.00000.2.wav,66149,0.346815,0.092243,0.132003,0.004620,1788.539719,111407.437613,2084.565132,75124.921716,...,4.806280,67.336563,-1.768610,28.348579,2.378768,45.717648,-1.938424,53.050835,2.517375,33.105122
3,blues.00000.3.wav,66149,0.363639,0.086856,0.132565,0.002448,1655.289045,111952.284517,1960.039988,82913.639269,...,-1.359111,47.739452,-3.841155,28.337118,1.218588,34.770935,-3.580352,50.836224,3.630866,32.023678
4,blues.00000.4.wav,66149,0.335579,0.088129,0.143289,0.001701,1630.656199,79667.267654,1948.503884,60204.020268,...,2.092937,30.336359,0.664582,45.880913,1.689446,51.363583,-3.392489,26.738789,0.536961,29.146694
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9985,rock.00099.5.wav,66149,0.349126,0.080515,0.050019,0.000097,1499.083005,164266.886443,1718.707215,85931.574523,...,5.773784,42.485981,-9.094270,38.326839,-4.246976,31.049839,-5.625813,48.804092,1.818823,38.966969
9986,rock.00099.6.wav,66149,0.372564,0.082626,0.057897,0.000088,1847.965128,281054.935973,1906.468492,99727.037054,...,2.074155,32.415203,-12.375726,66.418587,-3.081278,54.414265,-11.960546,63.452255,0.428857,18.697033
9987,rock.00099.7.wav,66149,0.347481,0.089019,0.052403,0.000701,1346.157659,662956.246325,1561.859087,138762.841945,...,-1.005473,78.228149,-2.524483,21.778994,4.809936,25.980829,1.775686,48.582378,-0.299545,41.586990
9988,rock.00099.8.wav,66149,0.387527,0.084815,0.066430,0.000320,2084.515327,203891.039161,2018.366254,22860.992562,...,4.123402,28.323744,-5.363541,17.209942,6.462601,21.442928,2.354765,24.843613,0.675824,12.787750


In [28]:
from sklearn.preprocessing import StandardScaler
fit = StandardScaler()
X = fit.fit_transform(np.array(df.iloc[:,1:-1], dtype=float))
X

array([[ 0.        , -0.48780784,  0.64052047, ..., -0.51356204,
         0.12841417, -0.29178072],
       [ 0.        , -0.40314187,  0.13183473, ...,  1.01138445,
         1.27578001,  0.05642464],
       [ 0.        , -0.36169428,  0.7644909 , ..., -0.04624405,
         0.65390663, -0.52145798],
       ...,
       [ 0.        , -0.35433044,  0.42997426, ..., -0.15370124,
         0.11765485, -0.33882395],
       [ 0.        ,  0.0883611 , -0.00630133, ..., -0.72456977,
         0.30333409, -0.95893743],
       [ 0.        , -0.11321002,  0.19536324, ..., -0.37245283,
        -0.47495901, -0.55112155]])

In [30]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.33)

In [31]:
len(y_train)

6693

In [32]:
len(y_test)

3297

In [33]:
from keras import Sequential

In [37]:
def trainModel(model, epochs, optimizer):
    batch_size = 128
    model.compile(optimizer = optimizer, loss = 'sparse_categorical_crossentropy', metrics = 'accuracy')
    return model.fit(X_train, y_train, validation_data = (X_test, y_test), epochs=epochs, batch_size = batch_size)

In [38]:
def plotValidate(history):
    print("Validation Accuracy", max(history.history['val_accuracy']))
    pd.DataFrame(history.history).plot(figsize=(12,6))
    plt.show()

In [39]:
model = keras.models.Sequential([
        keras.layers.Dense(512, activation='relu', input_shape=(X_train.shape[1],)),
        keras.layers.Dropout(0.2),
        
        keras.layers.Dense(256,activation='relu'),
        keras.layers.Dropout(0.2),
    
        keras.layers.Dense(128,activation='relu'),
        keras.layers.Dropout(0.2),
    
        keras.layers.Dense(64,activation='relu'),
        keras.layers.Dropout(0.2),
        
        keras.layers.Dense(10, activation='softmax')
])

print(model.summary())
model_history = trainModel(model=model, epochs=600, optimizer='adam')

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_5 (Dense)             (None, 512)               30208     
                                                                 
 dropout_4 (Dropout)         (None, 512)               0         
                                                                 
 dense_6 (Dense)             (None, 256)               131328    
                                                                 
 dropout_5 (Dropout)         (None, 256)               0         
                                                                 
 dense_7 (Dense)             (None, 128)               32896     
                                                                 
 dropout_6 (Dropout)         (None, 128)               0         
                                                                 
 dense_8 (Dense)             (None, 64)               

53/53 [==============================] - 0s 8ms/step - loss: 0.0721 - accuracy: 0.9792 - val_loss: 0.3374 - val_accuracy: 0.9139
Epoch 48/600
53/53 [==============================] - 0s 8ms/step - loss: 0.0661 - accuracy: 0.9795 - val_loss: 0.3560 - val_accuracy: 0.9126
Epoch 49/600
53/53 [==============================] - 0s 8ms/step - loss: 0.0699 - accuracy: 0.9776 - val_loss: 0.3615 - val_accuracy: 0.9072
Epoch 50/600
53/53 [==============================] - 0s 8ms/step - loss: 0.0718 - accuracy: 0.9776 - val_loss: 0.3630 - val_accuracy: 0.9072
Epoch 51/600
53/53 [==============================] - 0s 7ms/step - loss: 0.0826 - accuracy: 0.9737 - val_loss: 0.3378 - val_accuracy: 0.9120
Epoch 52/600
53/53 [==============================] - 0s 9ms/step - loss: 0.0598 - accuracy: 0.9828 - val_loss: 0.3317 - val_accuracy: 0.9172
Epoch 53/600
53/53 [==============================] - 0s 8ms/step - loss: 0.0572 - accuracy: 0.9813 - val_loss: 0.3648 - val_accuracy: 0.9084
Epoch 54/600
53/53 

Epoch 105/600
53/53 [==============================] - 0s 9ms/step - loss: 0.0324 - accuracy: 0.9889 - val_loss: 0.3753 - val_accuracy: 0.9139
Epoch 106/600
53/53 [==============================] - 0s 9ms/step - loss: 0.0363 - accuracy: 0.9873 - val_loss: 0.3834 - val_accuracy: 0.9178
Epoch 107/600
53/53 [==============================] - 0s 9ms/step - loss: 0.0336 - accuracy: 0.9906 - val_loss: 0.3879 - val_accuracy: 0.9148
Epoch 108/600
53/53 [==============================] - 1s 10ms/step - loss: 0.0260 - accuracy: 0.9912 - val_loss: 0.3999 - val_accuracy: 0.9160
Epoch 109/600
53/53 [==============================] - 1s 10ms/step - loss: 0.0357 - accuracy: 0.9889 - val_loss: 0.3843 - val_accuracy: 0.9178
Epoch 110/600
53/53 [==============================] - 0s 9ms/step - loss: 0.0316 - accuracy: 0.9898 - val_loss: 0.3966 - val_accuracy: 0.9151
Epoch 111/600
53/53 [==============================] - 1s 9ms/step - loss: 0.0440 - accuracy: 0.9851 - val_loss: 0.3918 - val_accuracy: 0.91

53/53 [==============================] - 0s 9ms/step - loss: 0.0243 - accuracy: 0.9925 - val_loss: 0.3841 - val_accuracy: 0.9233
Epoch 163/600
53/53 [==============================] - 0s 9ms/step - loss: 0.0247 - accuracy: 0.9925 - val_loss: 0.4060 - val_accuracy: 0.9196
Epoch 164/600
53/53 [==============================] - 0s 9ms/step - loss: 0.0134 - accuracy: 0.9958 - val_loss: 0.4134 - val_accuracy: 0.9196
Epoch 165/600
53/53 [==============================] - 0s 9ms/step - loss: 0.0180 - accuracy: 0.9931 - val_loss: 0.4133 - val_accuracy: 0.9202
Epoch 166/600
53/53 [==============================] - 0s 9ms/step - loss: 0.0144 - accuracy: 0.9955 - val_loss: 0.4117 - val_accuracy: 0.9254
Epoch 167/600
53/53 [==============================] - 0s 9ms/step - loss: 0.0199 - accuracy: 0.9933 - val_loss: 0.4241 - val_accuracy: 0.9208
Epoch 168/600
53/53 [==============================] - 0s 9ms/step - loss: 0.0233 - accuracy: 0.9924 - val_loss: 0.4115 - val_accuracy: 0.9217
Epoch 169/600

53/53 [==============================] - 0s 9ms/step - loss: 0.0184 - accuracy: 0.9937 - val_loss: 0.4375 - val_accuracy: 0.9275
Epoch 220/600
53/53 [==============================] - 0s 9ms/step - loss: 0.0222 - accuracy: 0.9921 - val_loss: 0.4632 - val_accuracy: 0.9217
Epoch 221/600
53/53 [==============================] - 0s 9ms/step - loss: 0.0261 - accuracy: 0.9933 - val_loss: 0.4101 - val_accuracy: 0.9208
Epoch 222/600
53/53 [==============================] - 0s 9ms/step - loss: 0.0158 - accuracy: 0.9949 - val_loss: 0.4365 - val_accuracy: 0.9257
Epoch 223/600
53/53 [==============================] - 0s 9ms/step - loss: 0.0205 - accuracy: 0.9940 - val_loss: 0.4281 - val_accuracy: 0.9214
Epoch 224/600
53/53 [==============================] - 0s 9ms/step - loss: 0.0237 - accuracy: 0.9924 - val_loss: 0.4403 - val_accuracy: 0.9211
Epoch 225/600
53/53 [==============================] - 0s 9ms/step - loss: 0.0301 - accuracy: 0.9916 - val_loss: 0.4537 - val_accuracy: 0.9227
Epoch 226/600

...

53/53 [==============================] - 0s 9ms/step - loss: 0.0242 - accuracy: 0.9912 - val_loss: 0.4397 - val_accuracy: 0.9251
Epoch 334/600
53/53 [==============================] - 0s 9ms/step - loss: 0.0180 - accuracy: 0.9940 - val_loss: 0.4094 - val_accuracy: 0.9248
Epoch 335/600
53/53 [==============================] - 0s 9ms/step - loss: 0.0187 - accuracy: 0.9937 - val_loss: 0.4156 - val_accuracy: 0.9239
Epoch 336/600
53/53 [==============================] - 0s 9ms/step - loss: 0.0139 - accuracy: 0.9952 - val_loss: 0.4370 - val_accuracy: 0.9263
Epoch 337/600
53/53 [==============================] - 1s 10ms/step - loss: 0.0158 - accuracy: 0.9951 - val_loss: 0.4365 - val_accuracy: 0.9284
Epoch 338/600
53/53 [==============================] - 0s 9ms/step - loss: 0.0125 - accuracy: 0.9957 - val_loss: 0.4364 - val_accuracy: 0.9257
Epoch 339/600
53/53 [==============================] - 0s 9ms/step - loss: 0.0198 - accuracy: 0.9945 - val_loss: 0.4604 - val_accuracy: 0.9224
Epoch 340/60

53/53 [==============================] - 0s 9ms/step - loss: 0.0142 - accuracy: 0.9949 - val_loss: 0.4192 - val_accuracy: 0.9251
Epoch 391/600
53/53 [==============================] - 0s 9ms/step - loss: 0.0114 - accuracy: 0.9967 - val_loss: 0.4229 - val_accuracy: 0.9281
Epoch 392/600
53/53 [==============================] - 0s 9ms/step - loss: 0.0139 - accuracy: 0.9957 - val_loss: 0.4925 - val_accuracy: 0.9248
Epoch 393/600
53/53 [==============================] - 1s 10ms/step - loss: 0.0139 - accuracy: 0.9955 - val_loss: 0.4729 - val_accuracy: 0.9299
Epoch 394/600
53/53 [==============================] - 1s 10ms/step - loss: 0.0094 - accuracy: 0.9973 - val_loss: 0.5161 - val_accuracy: 0.9242
Epoch 395/600
53/53 [==============================] - 1s 10ms/step - loss: 0.0171 - accuracy: 0.9942 - val_loss: 0.4663 - val_accuracy: 0.9230
Epoch 396/600
53/53 [==============================] - 1s 10ms/step - loss: 0.0138 - accuracy: 0.9958 - val_loss: 0.4810 - val_accuracy: 0.9227
Epoch 397

53/53 [==============================] - 1s 10ms/step - loss: 0.0211 - accuracy: 0.9931 - val_loss: 0.4653 - val_accuracy: 0.9217
Epoch 448/600
53/53 [==============================] - 0s 9ms/step - loss: 0.0117 - accuracy: 0.9960 - val_loss: 0.4846 - val_accuracy: 0.9272
Epoch 449/600
53/53 [==============================] - 0s 9ms/step - loss: 0.0125 - accuracy: 0.9952 - val_loss: 0.4709 - val_accuracy: 0.9266
Epoch 450/600
53/53 [==============================] - 0s 9ms/step - loss: 0.0125 - accuracy: 0.9964 - val_loss: 0.4827 - val_accuracy: 0.9242
Epoch 451/600
53/53 [==============================] - 0s 9ms/step - loss: 0.0126 - accuracy: 0.9963 - val_loss: 0.5000 - val_accuracy: 0.9254
Epoch 452/600
53/53 [==============================] - 0s 9ms/step - loss: 0.0125 - accuracy: 0.9960 - val_loss: 0.4763 - val_accuracy: 0.9254
Epoch 453/600
53/53 [==============================] - 0s 9ms/step - loss: 0.0122 - accuracy: 0.9961 - val_loss: 0.4985 - val_accuracy: 0.9284
Epoch 454/60

53/53 [==============================] - 1s 10ms/step - loss: 0.0164 - accuracy: 0.9952 - val_loss: 0.4778 - val_accuracy: 0.9263
Epoch 505/600
53/53 [==============================] - 1s 10ms/step - loss: 0.0160 - accuracy: 0.9966 - val_loss: 0.4918 - val_accuracy: 0.9305
Epoch 506/600
53/53 [==============================] - 0s 9ms/step - loss: 0.0110 - accuracy: 0.9970 - val_loss: 0.4485 - val_accuracy: 0.9321
Epoch 507/600
53/53 [==============================] - 0s 9ms/step - loss: 0.0133 - accuracy: 0.9957 - val_loss: 0.4777 - val_accuracy: 0.9296
Epoch 508/600
53/53 [==============================] - 0s 9ms/step - loss: 0.0095 - accuracy: 0.9969 - val_loss: 0.4847 - val_accuracy: 0.9284
Epoch 509/600
53/53 [==============================] - 1s 9ms/step - loss: 0.0086 - accuracy: 0.9970 - val_loss: 0.4718 - val_accuracy: 0.9315
Epoch 510/600
53/53 [==============================] - 0s 9ms/step - loss: 0.0109 - accuracy: 0.9961 - val_loss: 0.4681 - val_accuracy: 0.9299
Epoch 511/6

53/53 [==============================] - 1s 9ms/step - loss: 0.0120 - accuracy: 0.9966 - val_loss: 0.4948 - val_accuracy: 0.9290
Epoch 562/600
53/53 [==============================] - 0s 9ms/step - loss: 0.0096 - accuracy: 0.9963 - val_loss: 0.5039 - val_accuracy: 0.9348
Epoch 563/600
53/53 [==============================] - 0s 9ms/step - loss: 0.0133 - accuracy: 0.9963 - val_loss: 0.4490 - val_accuracy: 0.9324
Epoch 564/600
53/53 [==============================] - 0s 9ms/step - loss: 0.0138 - accuracy: 0.9945 - val_loss: 0.4556 - val_accuracy: 0.9311
Epoch 565/600
53/53 [==============================] - 0s 9ms/step - loss: 0.0152 - accuracy: 0.9961 - val_loss: 0.4710 - val_accuracy: 0.9257
Epoch 566/600
53/53 [==============================] - 0s 9ms/step - loss: 0.0192 - accuracy: 0.9958 - val_loss: 0.4227 - val_accuracy: 0.9324
Epoch 567/600
53/53 [==============================] - 0s 9ms/step - loss: 0.0109 - accuracy: 0.9961 - val_loss: 0.4321 - val_accuracy: 0.9336
Epoch 568/600

In [47]:
test_loss, test_acc = model.evaluate(X_test, y_test, batch_size=128)
print("Test loss is:", test_loss)
print("The best test accuracy is:", test_acc*100)

26/26 [==============================] - 0s 9ms/step - loss: 0.5259 - accuracy: 0.9281
Test loss is: 0.5258957147598267
The best test accuracy is: 92.81164407730103
